# Cellular Automata

> *"The universe is a giant cellular automaton."* — Stephen Wolfram

## What is a Cellular Automaton?

A Cellular Automaton is a discrete model studied in computability theory, mathematics, physics, and theoretical biology. It consists of a regular grid of *cells*, each in one of a finite number of *states* (e.g., On/Off, Alive/Dead, Empty/Occupied). Time is also discrete, and the state of a cell at time $t+1$ is determined by a set of rules based on the states of its neighboring cells at time $t$.

Cellular automata are useful because:

- Local rules can generate complex emergent behavior
- They are naturally parallelizable
- They connect microscopic and macroscopic descriptions
- They model transport, fluids, diffusion, traffic, biology, and complex systems
  
Examples of Cellular Automata Applications
- Magnetism (Ising-like dynamics)
- Forest fires
- Epidemics
- Fluid simulation
- Traffic flow
- Pattern formation
- Crystal growth
- Reaction-diffusion systems
- Biological morphogenesis
- Artificial life


### Brief History and Important Contributions

| Date | Authors | Description |
| :---: | :--- | :--- |
| 1940s | von Neumann & Ulam | Developed the concept while at Los Alamos National Laboratory to study crystal growth and self-replicating systems. |
| 1970 | John H. Conway | Game of Life (Gardner, Scientific American) |
| 1983 | Stephen Wolfram | Conducted a systematic study of 1D cellular automata, classifying them into four complexity classes (Wolfram's Rule 30 is a famous pseudo-random number generator). Elementary CAs — 256 rules; Class I–IV taxonomy |
| 1986 | Frisch, Hasslacher, Pomeau (FHP) | Lattice Gas Automaton $\rightarrow$ Navier–Stokes |
| 1992 | Chen & Doolen | Lattice Boltzmann Method (LBM) |
| 2002 | Wolfram | "A New Kind of Science" |
| 2010s+ | N/A | CA for ML, quantum computing, biology |

**Key references and links:**
- [von Neumann's original manuscript (digitised)](https://archive.org/details/theoryofselfrepr00vonn_0)
- [Wolfram's Elementary CA explorer](https://www.wolframalpha.com/input?i=Rule+110)
- [LifeWiki — the Game of Life encyclopædia](https://conwaylife.com/wiki/)
- [Golly — open-source CA simulator](https://golly.sourceforge.net/)


### Cellular Automata vs. Agent-Based Simulations
While both CA and Agent-Based Models (ABM) simulate complex systems, they have distinct paradigms:
* **Cellular Automata:** Space is the primary entity. Cells are fixed in a grid, and *states* move between cells. Rules are completely local and uniform. 
* **Agent-Based Modeling:** Agents are the primary entities. They can move freely in continuous or discrete space, have heterogeneous internal states (memory, goals), and can interact non-locally. 
* **Physics Application:** CA is typically used for continuous media (fluids, heat diffusion, quantum fields), while ABMs are used for granular matter, active matter, or biological swarms.

## Important Modern Extensions

### Lattice Gas Cellular Automata (LGCA)

Discrete particle models for fluids.

### Lattice Boltzmann Method (LBM)

Widely used modern CFD method derived from kinetic theory.

Applications include:

- Turbulence
- Multiphase flows
- Porous media
- Microfluidics
- Blood flow

## Elementary Cellular Automata

A one-dimensional binary CA consists of:

- States: $0$ or $1$
- Neighborhood size: 3 cells
- Rule applied simultaneously

There are:

$$
2^{2^3} = 256
$$

possible rules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def evolve(rule_number, steps=100, size=201):
    rule_bin = np.array([int(x) for x in np.binary_repr(rule_number, width=8)])

    grid = np.zeros((steps, size), dtype=int)
    grid[0, size // 2] = 1

    for t in range(steps - 1):
        for i in range(1, size - 1):
            neighborhood = grid[t, i-1:i+2]
            idx = 7 - int("".join(map(str, neighborhood)), 2)
            grid[t+1, i] = rule_bin[idx]

    return grid

rule = 30
grid = evolve(rule)

plt.figure(figsize=(10, 10))
plt.imshow(grid, cmap="binary", interpolation="nearest")
plt.title(f"Elementary Cellular Automaton — Rule {rule}")
plt.xlabel("Cell")
plt.ylabel("Time")
plt.show()

### Wolfram Classification

Stephen Wolfram classified CA into four categories:

| Class | Behavior |
|---|---|
| I | Homogeneous |
| II | Periodic |
| III | Chaotic |
| IV | Complex / edge of chaos |

Rule 30 is chaotic.  
Rule 110 is computationally universal.

In [ ]:
# ── Elementary 1-D Cellular Automaton ───────────────────────────────────────

def elementary_ca(rule_number: int, n_cells: int = 201, n_steps: int = 100) -> np.ndarray:
    """Simulate a 1-D elementary CA.

    Parameters
    ----------
    rule_number : int
        Wolfram rule index (0–255).
    n_cells : int
        Width of the lattice.
    n_steps : int
        Number of time steps.

    Returns
    -------
    np.ndarray  shape (n_steps + 1, n_cells), dtype uint8
    """
    # Decode rule number into a lookup table
    rule_bits = np.array([(rule_number >> k) & 1 for k in range(8)], dtype=np.uint8)

    grid = np.zeros((n_steps + 1, n_cells), dtype=np.uint8)
    grid[0, n_cells // 2] = 1  # single seed in the centre

    for t in range(n_steps):
        left   = np.roll(grid[t], 1)
        centre = grid[t]
        right  = np.roll(grid[t], -1)
        # neighbourhood index k ∈ {0,…,7}
        k = 4 * left + 2 * centre + right
        grid[t + 1] = rule_bits[k]

    return grid


# Plot four representative rules side-by-side
rules_to_show = [30, 90, 110, 184]
fig, axes = plt.subplots(1, 4, figsize=(14, 5))

for ax, rule in zip(axes, rules_to_show):
    grid = elementary_ca(rule, n_cells=201, n_steps=100)
    ax.imshow(grid, cmap='binary', interpolation='nearest', aspect='auto')
    ax.set_title(f'Rule {rule}', fontsize=12)
    ax.set_xlabel('Cell index')
    ax.set_ylabel('Time step')

fig.suptitle('Wolfram Elementary Cellular Automata', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print("Rule 30 — chaotic (used in Mathematica's RNG!)")
print("Rule 110 — proved Turing-complete by Matthew Cook (2004)")

In [ ]:
# ── Standard imports used throughout this notebook ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Optional: ipywidgets for interactive controls
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, fixed
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not found – interactive sliders will be skipped.")
    print("Install with:  pip install ipywidgets")

# Consistent random seed for reproducibility
RNG = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'serif',
})
print("Imports OK ✓")

In [ ]:
# ── Interactive widget: explore all 256 rules ────────────────────────────────
if WIDGETS_AVAILABLE:
    def plot_rule(rule):
        grid = elementary_ca(rule, n_cells=201, n_steps=100)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.imshow(grid, cmap='binary', interpolation='nearest', aspect='auto')
        ax.set_title(f'Wolfram Rule {rule}', fontsize=13)
        ax.set_xlabel('Cell')
        ax.set_ylabel('Time')
        plt.tight_layout()
        plt.show()

    interact(plot_rule, rule=widgets.IntSlider(min=0, max=255, step=1, value=110,
                                               description='Rule:', continuous_update=False))
else:
    print("Install ipywidgets to explore all 256 rules interactively.")

### Exercises
#### Rule 110
Implement rule 110. Is it chaotic? why?

## Conway's Game of Life

Invented by mathematician John Conway in 1970, this is a 2D cellular automaton where cells can be ALIVE (1) or DEAD (0). The neighborhood consists of the 8 surrounding cells (Moore neighborhood).

The rules are simple:
1. **Underpopulation:** A live cell with fewer than 2 live neighbors dies.
2. **Survival:** A live cell with 2 or 3 live neighbors lives on.
3. **Overpopulation:** A live cell with more than 3 live neighbors dies.
4. **Reproduction:** A dead cell with exactly 3 live neighbors becomes a live cell.

Despite its simplicity, the Game of Life is **Turing Complete**, meaning it can simulate any computer algorithm. 

**Extreme Uses of Game of Life:**
* [Building a digital clock in GoL](https://codegolf.stackexchange.com/questions/88783/build-a-digital-clock-in-conways-game-of-life)
* [A Game of Life simulator programmed *inside* the Game of Life](https://www.youtube.com/watch?v=xP5-iIeKXE8)
* [Tetris implemented in Game of Life](https://codegolf.stackexchange.com/questions/11880/build-a-working-game-of-tetris-in-conways-game-of-life)

### Famous Structures

### Still Lifes
- Block
- Beehive

### Oscillators
- Blinker
- Toad
- Pulsar

### Spaceships
- Glider
- Lightweight spaceship (LWSS)

### Complex Structures
- Glider guns
- Breeders
- Self-replicators

---

### Extreme Constructions

Some groups have built:

- Logic gates
- Adders
- Multipliers
- Entire computers
- Turing machines

Notable examples:

- Gosper Glider Gun
- OTCA Metapixel
- Gemini self-replicator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import scipy.signal

# Good Practice: Use SciPy's 2D convolution for highly optimized grid updates instead of nested loops
def update_life(grid):
    """Calculates the next state of the Conway's Game of Life grid."""
    # Define the Moore neighborhood kernel
    kernel = np.array([[1, 1, 1],
                       [1, 0, 1],
                       [1, 1, 1]])
    
    # Count neighbors using 2D convolution with boundary wrap-around (toroidal grid)
    neighbors = scipy.signal.convolve2d(grid, kernel, mode='same', boundary='wrap')
    
    # Apply Conway's rules
    birth = (neighbors == 3) & (grid == 0)
    survive = ((neighbors == 2) | (neighbors == 3)) & (grid == 1)
    
    # Create the new grid
    new_grid = np.zeros_like(grid)
    new_grid[birth | survive] = 1
    return new_grid

# Setup the grid
grid_size = 50
np.random.seed(42)
# Initialize with 20% alive cells
initial_grid = np.random.choice([0, 1], size=(grid_size, grid_size), p=[0.8, 0.2])

# Animation setup
fig, ax = plt.subplots(figsize=(5, 5))
mat = ax.matshow(initial_grid, cmap='binary')
ax.axis('off')

def animate(frame, grid_ref):
    grid_ref[0] = update_life(grid_ref[0])
    mat.set_data(grid_ref[0])
    return [mat]

# We pass the grid in a list to maintain a mutable reference
anim = FuncAnimation(fig, animate, fargs=([initial_grid.copy()],), frames=100, interval=100, blit=True)
plt.close() # Prevent static plot from showing

# Display in Jupyter Notebook
HTML(anim.to_jshtml())

### Exercises for Game of Life
1. **Glider:** Implement a glider and show that it moves
2. **Glider Gun:** Modify the initialization code above to place a "Gosper Glider Gun" manually instead of random noise. Observe how it generates moving structures (gliders) indefinitely.
3. **Boundary Conditions:** The current code uses a toroidal boundary (`boundary='wrap'`). Change this to a hard wall (`boundary='fill', fillvalue=0`) and observe how it affects patterns hitting the edge.
4. **Density Tracking:** Create an array to track the fraction of living cells over time and plot it. Does the system reach a steady state, periodic oscillations, or die out?
5. **(Entropy):** Define a local entropy $H(t) = -\sum_k p_k \log p_k$ where $p_k$ is the fraction of $3\times 3$ blocks in each of their $2^9$ states. Plot $H(t)$ for the R-pentomino. What does the asymptotic value tell you?

## Modeling Traffic: The Nagel-Schreckenberg Model


```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/1/19/Nagel-schreck_rho%3D0.35_p%3D0.3.png/500px-Nagel-schreck_rho%3D0.35_p%3D0.3.png
```
Cellular Automata are heavily used in non-equilibrium statistical mechanics. A famous example is the **Nagel-Schreckenberg (NaSch) model** (1992) for vehicular traffic flow. It shows how traffic jams can emerge from nowhere (phantom jams) simply due to delayed human reaction times.

Consider a 1D grid of $L$ cells with periodic boundaries. Each cell is empty or contains a car with velocity $v \in \{0, 1, ..., v_{max}\}$. 

For each time step $t$, the rules are applied to all cars simultaneously:
1. **Acceleration:** If $v < v_{max}$, the car accelerates: $v \to v + 1$.
2. **Slowing down:** If the distance to the next car ahead is $d$, and $v \ge d$, the driver brakes to avoid a crash: $v \to d - 1$.
3. **Randomization:** With a probability $p$ (modeling human hesitation/over-braking), if $v > 0$, the velocity is reduced: $v \to v - 1$.
4. **Movement:** The car moves forward by $v$ cells.

We can visualize this using a **Space-Time Diagram**, where the x-axis is space and the y-axis is time.

In [ ]:
def simulate_traffic(length=100, density=0.3, v_max=5, p_slow=0.3, steps=100):
    """Simulates the Nagel-Schreckenberg traffic model."""
    # -1 represents an empty road cell, >=0 represents a car's velocity
    road = np.full(length, -1)
    
    # Place cars randomly
    num_cars = int(length * density)
    positions = np.random.choice(length, num_cars, replace=False)
    road[positions] = np.random.randint(0, v_max + 1, size=num_cars)
    
    history = np.zeros((steps, length))
    
    for t in range(steps):
        history[t] = (road != -1).astype(int) # Save binary state for visualization
        
        # Find car indices
        car_idx = np.where(road != -1)[0]
        if len(car_idx) == 0:
            break
            
        new_road = np.full(length, -1)
        
        for i in car_idx:
            v = road[i]
            
            # Find distance to next car
            next_i = (i + 1) % length
            d = 1
            while road[next_i] == -1 and d <= v_max:
                d += 1
                next_i = (next_i + 1) % length
                
            # Rule 1: Accelerate
            if v < v_max:
                v += 1
            # Rule 2: Slow down
            if v >= d:
                v = d - 1
            # Rule 3: Randomization
            if v > 0 and np.random.rand() < p_slow:
                v -= 1
                
            # Rule 4: Move
            new_pos = (i + v) % length
            new_road[new_pos] = v
            
        road = new_road
        
    return history

# Run simulation
time_steps = 100
road_length = 150
traffic_history = simulate_traffic(length=road_length, density=0.25, v_max=5, p_slow=0.3, steps=time_steps)

# Visualize Space-Time Diagram
plt.figure(figsize=(10, 6))
plt.imshow(traffic_history, cmap='binary', aspect='auto', interpolation='none')
plt.title("Nagel-Schreckenberg Traffic Model: Space-Time Diagram")
plt.xlabel("Space (Road Position)")
plt.ylabel("Time (Moving Downwards)")
plt.show()

# Notice the backward-propagating black waves: these are phantom traffic jams!

### Exercises
#### Braking probability 
Vary the braking probability $p \in [0, 1]$ and plot the critical density (flow maximum) as a function of $p$. 
#### Bottleneck 
Add a *bottleneck* (a single cell with reduced $v_{\max} = 1$). Observe how upstream jams form and compute the fraction of time each car spends stationary.
#### Fundamental diagrams
Compute the fundamental diagrams for the previous examples
#### Random influence
Change the random parameter and visualize if there is or not any traffic jam
#### Animation on a street
Perform an animation of traffic using vpython
#### (Research) 
The NS model predicts jam speeds of about $-15$ km/h (backwards). Real traffic jams on German autobahns propagate at approximately $-15$ km/h. Reproduce this result by choosing physically realistic parameters ($L$ = cell size in metres, time step in seconds).

## Diffusion, Lattice Gases, and the Chapman-Enskog Expansion

How do we connect discrete microscopic CA rules to continuous macroscopic physical equations (like the Navier-Stokes equations for fluids or the Diffusion equation)? 

This transition is formalized using the **Chapman-Enskog Expansion**, a multi-scale perturbation technique originally developed for the Boltzmann equation.

### A Simple 1D Diffusion CA
Consider a macroscopic diffusion equation:
$$ \frac{\partial \rho}{\partial t} = D \frac{\partial^2 \rho}{\partial x^2} $$

We can model this using a probabilistic CA (a random walk). Let $p(x,t)$ be the probability of a particle being at cell $x$ at time $t$. In a simple 1D lattice, a particle jumps left or right with equal probability.
The microscopic rule for the probability density $\rho$ is:

$$ \rho(x, t+\Delta t) = \frac{1}{2}\rho(x-\Delta x, t) + \frac{1}{2}\rho(x+\Delta x, t) $$

### The Chapman-Enskog Concept (Simplified Limit)
To see how the CA maps to the continuous PDE, we apply a Taylor series expansion in space and time (assuming $\Delta x$ and $\Delta t$ are very small):

1. **LHS (Time expansion):** 
   $$ \rho(x, t+\Delta t) \approx \rho(x, t) + \frac{\partial \rho}{\partial t}\Delta t $$
2. **RHS (Space expansion):** 
   $$ \rho(x \pm \Delta x, t) \approx \rho(x,t) \pm \frac{\partial \rho}{\partial x}\Delta x + \frac{1}{2}\frac{\partial^2 \rho}{\partial x^2}(\Delta x)^2 $$

Substitute the RHS back into the rule:
$$ \frac{1}{2}\left[ \rho - \frac{\partial \rho}{\partial x}\Delta x + \frac{1}{2}\frac{\partial^2 \rho}{\partial x^2}\Delta x^2 \right] + \frac{1}{2}\left[ \rho + \frac{\partial \rho}{\partial x}\Delta x + \frac{1}{2}\frac{\partial^2 \rho}{\partial x^2}\Delta x^2 \right] $$
$$ = \rho(x,t) + \frac{1}{2}\frac{\partial^2 \rho}{\partial x^2}\Delta x^2 $$

Equating LHS and RHS:
$$ \rho + \frac{\partial \rho}{\partial t}\Delta t = \rho + \frac{1}{2}\frac{\partial^2 \rho}{\partial x^2}\Delta x^2 $$
$$ \frac{\partial \rho}{\partial t} = \left( \frac{\Delta x^2}{2 \Delta t} \right) \frac{\partial^2 \rho}{\partial x^2} $$

By defining our macroscopic diffusion coefficient as $D = \frac{\Delta x^2}{2\Delta t}$, we have successfully deduced the macroscopic diffusion equation from a local CA rule!

*Note: In modern fluid dynamics, the Lattice Boltzmann Method (LBM) utilizes the full Chapman-Enskog expansion, expanding the particle distribution functions across different timescales to recover the Navier-Stokes equations.*

In [ ]:
# Simulating 1D Diffusion using a probabilistic Cellular Automaton 
def simulate_diffusion(length=100, steps=200, particles=5000):
    """Simulates 1D diffusion using thousands of random walkers on a discrete lattice."""
    # Place all particles in the middle of the lattice
    grid = np.zeros(length)
    center = length // 2
    grid[center] = particles
    
    history = np.zeros((steps, length))
    
    for t in range(steps):
        history[t] = grid
        new_grid = np.zeros(length)
        
        # For every cell, distribute its particles randomly to the left and right
        for x in range(1, length - 1):
            if grid[x] > 0:
                # Binomial distribution models the random left/right split
                move_right = np.random.binomial(grid[x], 0.5)
                move_left = grid[x] - move_right
                
                new_grid[x+1] += move_right
                new_grid[x-1] += move_left
                
        grid = new_grid
        
    return history

diff_history = simulate_diffusion()

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
x_vals = np.arange(100)
line, = ax.plot(x_vals, diff_history[0], color='blue')

ax.set_ylim(0, np.max(diff_history) * 1.1)
ax.set_title("1D Diffusion Cellular Automaton")
ax.set_xlabel("Lattice Position")
ax.set_ylabel("Particle Count")

def animate_diff(frame):
    line.set_ydata(diff_history[frame])
    return [line]

anim_diff = FuncAnimation(fig, animate_diff, frames=200, interval=40, blit=True)
plt.close()

HTML(anim_diff.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 200
particles = np.zeros(L)

particles[L//2] = 1000

history = []

for _ in range(200):

    new_particles = np.zeros(L)

    for i in range(L):

        n = int(particles[i])

        left = np.random.binomial(n, 0.5)
        right = n - left

        new_particles[(i-1)%L] += left
        new_particles[(i+1)%L] += right

    particles = new_particles
    history.append(particles.copy())

history = np.array(history)

plt.figure(figsize=(10,6))
plt.imshow(history, aspect='auto')
plt.colorbar(label='Particle density')
plt.xlabel("Position")
plt.ylabel("Time")
plt.title("Diffusion Cellular Automaton")
plt.show()

### Exercises
#### 2D Diffusion ceullar automata
- implement a 2D diffusion cellular automata. Use the same $p$ for each direction. Plot the mean squered displacement as a function of time. Is it proportional to time or not? Now use slightly different prob for a given direction, this favors that diffusion dynamics. What happens now?

## Lattice Boltzmann Method

### From Lattice Gas to Boltzmann

The **Lattice Gas Automaton (LGA)** of Frisch, Hasslacher & Pomeau (1986) simulated fluid dynamics with Boolean particles on a hexagonal lattice. The key insight: a microdynamical equation with the right symmetry group **coarse-grains** to the Navier–Stokes equations.

The **Lattice Boltzmann Method (LBM)** replaces the Boolean particle populations $n_i \in \{0,1\}$ with real-valued **distribution functions** $f_i(\mathbf{x}, t) \geq 0$ on a discrete velocity set $\{\mathbf{e}_i\}$.

### The LBM Equation

On the **D2Q9** lattice (2D, 9 velocities), the update is split into:

**Collision** (BGK approximation):
$$f_i^*(\mathbf{x}, t) = f_i(\mathbf{x}, t) - \frac{\Delta t}{\tau}\bigl[f_i(\mathbf{x}, t) - f_i^{\text{eq}}(\mathbf{x}, t)\bigr]$$

**Streaming**:
$$f_i(\mathbf{x} + \mathbf{e}_i \Delta t, \; t + \Delta t) = f_i^*(\mathbf{x}, t)$$

The **Maxwell–Boltzmann equilibrium** (expanded to second order in $u/c_s$):
$$f_i^{\text{eq}} = w_i \rho \left[1 + \frac{\mathbf{e}_i \cdot \mathbf{u}}{c_s^2} + \frac{(\mathbf{e}_i \cdot \mathbf{u})^2}{2c_s^4} - \frac{|\mathbf{u}|^2}{2c_s^2}\right]$$

where $c_s^2 = 1/3$ is the lattice speed of sound and $w_i$ are lattice weights.

**Macroscopic quantities** (moments of $f_i$):
$$\rho = \sum_i f_i, \qquad \rho \mathbf{u} = \sum_i f_i \mathbf{e}_i$$

A Chapman–Enskog expansion (to 2nd order in Knudsen number $\varepsilon = \Delta t / \tau$) recovers the **incompressible Navier–Stokes equations** with kinematic viscosity:
$$\boxed{\nu = c_s^2 \left(\tau - \frac{\Delta t}{2}\right)}$$

### Links to Modern LBM Resources
- [Palabos — open-source LBM library (C++)](https://palabos.unige.ch/)
- [PyLBM — Python LBM](https://pylbm.readthedocs.io/)
- [Kruger et al., *The Lattice Boltzmann Method* (Springer, free PDF)](https://link.springer.com/book/10.1007/978-3-319-44649-3)

In [ ]:
# ── Minimal D2Q9 Lattice Boltzmann: lid-driven cavity / Kármán vortex street ─
# Adapted from Philip Mocz's elegant implementation.
# Reference: https://philip-mocz.medium.com/

# D2Q9 lattice vectors and weights
e = np.array([
    [0,  0],
    [1,  0], [-1,  0], [0,  1], [0, -1],
    [1,  1], [-1,  1], [1, -1], [-1, -1]
], dtype=float)   # shape (9, 2)

w = np.array([4/9,
              1/9,  1/9,  1/9,  1/9,
              1/36, 1/36, 1/36, 1/36])  # shape (9,)

cs2 = 1.0 / 3.0   # lattice speed of sound squared


def equilibrium(rho: np.ndarray, ux: np.ndarray,
                uy: np.ndarray) -> np.ndarray:
    """Compute D2Q9 equilibrium distribution.

    Returns array of shape (9, Ny, Nx).
    """
    usq = ux**2 + uy**2                     # shape (Ny, Nx)
    feq = np.zeros((9, *rho.shape))
    for i in range(9):
        eu = e[i, 0] * ux + e[i, 1] * uy   # e_i · u
        feq[i] = w[i] * rho * (
            1.0 + eu / cs2 + eu**2 / (2 * cs2**2) - usq / (2 * cs2)
        )
    return feq


def lbm_step(f: np.ndarray, tau: float,
             obstacle: np.ndarray) -> tuple:
    """One BGK-LBM step: collision → streaming → bounce-back.

    Parameters
    ----------
    f        : distribution function, shape (9, Ny, Nx)
    tau      : relaxation time
    obstacle : bool array (Ny, Nx), True where solid

    Returns
    -------
    f, rho, ux, uy
    """
    # Macroscopic variables
    rho = f.sum(axis=0)
    ux  = (f * e[:, 0, np.newaxis, np.newaxis]).sum(axis=0) / rho
    uy  = (f * e[:, 1, np.newaxis, np.newaxis]).sum(axis=0) / rho

    # Collision (BGK)
    feq = equilibrium(rho, ux, uy)
    f_out = f - (f - feq) / tau

    # Streaming
    for i in range(9):
        f_out[i] = np.roll(f_out[i], int(e[i, 0]), axis=1)   # x
        f_out[i] = np.roll(f_out[i], int(e[i, 1]), axis=0)   # y

    # Halfway bounce-back on obstacle
    # Opposite direction indices for D2Q9
    opp = [0, 2, 1, 4, 3, 8, 7, 6, 5]
    for i in range(9):
        f_out[i, obstacle] = f[opp[i], obstacle]

    return f_out, rho, ux, uy


# ── Kármán vortex street setup ───────────────────────────────────────────────
Nx, Ny = 220, 80
tau = 0.6          # relaxation time → Re ≈ Ny/(3*tau*...) ≈ 80
u_in = 0.1         # inlet velocity

# Circular obstacle
cx, cy, cr = Nx//5, Ny//2, Ny//7
Y, X = np.mgrid[0:Ny, 0:Nx]
obstacle = (X - cx)**2 + (Y - cy)**2 < cr**2

# Initialise
rho0 = np.ones((Ny, Nx))
ux0  = u_in * np.ones((Ny, Nx))
uy0  = np.zeros((Ny, Nx))
f = equilibrium(rho0, ux0, uy0)

print(f"LBM grid: {Nx}×{Ny},  tau={tau},  u_in={u_in}")
print(f"Kinematic viscosity ν = {cs2*(tau - 0.5):.4f}")

In [ ]:
# ── Run LBM and animate vorticity ────────────────────────────────────────────
N_WARMUP  = 4000   # steps before recording
N_FRAMES  = 80
STEPS_PER_FRAME = 15

print(f"Warming up {N_WARMUP} steps …")
for _ in range(N_WARMUP):
    # Inlet: impose velocity on left column
    f[:, :, 0] = equilibrium(rho0[:, :1], ux0[:, :1], uy0[:, :1])[:, :, 0]
    f, rho, ux, uy = lbm_step(f, tau, obstacle)
print("Warm-up done. Collecting frames …")

# Vorticity: ω_z = ∂u_y/∂x − ∂u_x/∂y  (finite differences)
def vorticity(ux, uy):
    dvdx = np.roll(uy, -1, axis=1) - np.roll(uy, 1, axis=1)
    dudy = np.roll(ux, -1, axis=0) - np.roll(ux, 1, axis=0)
    return dvdx - dudy

frames_vort = []
for _ in range(N_FRAMES):
    for _ in range(STEPS_PER_FRAME):
        f[:, :, 0] = equilibrium(rho0[:, :1], ux0[:, :1], uy0[:, :1])[:, :, 0]
        f, rho, ux, uy = lbm_step(f, tau, obstacle)
    omega = vorticity(ux, uy)
    omega[obstacle] = np.nan
    frames_vort.append(omega.copy())

print(f"{N_FRAMES} frames collected. Building animation …")

fig_lbm, ax_lbm = plt.subplots(figsize=(11, 4))
vmax = 0.02
im_lbm = ax_lbm.imshow(frames_vort[0], cmap='RdBu_r',
                        vmin=-vmax, vmax=vmax, origin='lower',
                        interpolation='bilinear', animated=True)
plt.colorbar(im_lbm, ax=ax_lbm, label=r'Vorticity $\omega_z$')
ax_lbm.set_title('D2Q9 Lattice Boltzmann — Kármán Vortex Street')
ax_lbm.axis('off')

def update_lbm(i):
    im_lbm.set_data(frames_vort[i])
    return [im_lbm]

anim_lbm = animation.FuncAnimation(fig_lbm, update_lbm,
                                    frames=N_FRAMES, interval=80, blit=True)
plt.close(fig_lbm)
HTML(anim_lbm.to_jshtml())

## 5. Other Applications & Future Research Topics

Cellular automata are not just historical curiosities; they remain highly active areas of modern physics research.

**More Physics Applications:**
* **Ising Model mapping:** Certain cellular automata can compute the partition function and critical phenomena of Ising models.
* **Forest Fire Models:** Demonstrating Self-Organized Criticality (SOC) where a system naturally evolves toward a critical state, yielding fractal behavior and power-law distribution in fire sizes.
* **Epidemiology:** CA models easily integrate spatial topology into SIR (Susceptible-Infected-Recovered) models, representing how diseases spread across localized populations.
* **Reaction-Diffusion CA**: Pattern formation.
* **Cyclic Cellular Automata**: Spiral waves and excitable media.
* **Lattice Gas Automata**: Discrete kinetic theory.
* **Active matter**: Collective motion and flocking.
* **Self-Replication**: Artificial life.


### Future Research Frontiers
1. **Quantum Cellular Automata (QCA):** Extending CA rules to quantum amplitudes. QCA are being heavily researched as a framework for fault-tolerant quantum computation and modeling quantum field theories discretely.
2. **Graph Cellular Automata:** Moving away from rigid Cartesian grids, rules are applied over complex network topologies (e.g., social networks, neural connectomes).
3. **Neural Cellular Automata (NCA):** Integrating machine learning with CA. Researchers use deep learning to "learn" the specific update rules required to grow specific target structures (like organs in biological modeling) or to design self-repairing materials.



## Reaction–Diffusion: Turing Patterns

Turing (1952) showed that two interacting chemicals — an activator $u$ and an inhibitor $v$ — can spontaneously form spatial patterns from a homogeneous state:

$$\partial_t u = D_u \nabla^2 u + f(u,v), \qquad \partial_t v = D_v \nabla^2 v + g(u,v)$$

The **Gray–Scott model** (discrete CA approximation):

In [ ]:
# ── Gray–Scott Reaction–Diffusion (CA implementation) ────────────────────────

def laplacian2d(field: np.ndarray) -> np.ndarray:
    """2-D discrete Laplacian with periodic BC (5-point stencil)."""
    return (
        np.roll(field,  1, axis=0) + np.roll(field, -1, axis=0) +
        np.roll(field,  1, axis=1) + np.roll(field, -1, axis=1) -
        4 * field
    )


def gray_scott(N: int = 100, n_steps: int = 5000, dt: float = 1.0,
               Du: float = 0.16, Dv: float = 0.08,
               F: float = 0.035, k: float = 0.065,
               seed: int = 0) -> np.ndarray:
    """Simulate the Gray–Scott RD model.

    Returns the final `v` field (inhibitor concentration).
    """
    rng = np.random.default_rng(seed)
    u = np.ones((N, N))
    v = np.zeros((N, N))
    # Seed with a small perturbation patch
    r = N // 8
    u[N//2-r:N//2+r, N//2-r:N//2+r] = 0.5 + 0.02 * rng.standard_normal((2*r, 2*r))
    v[N//2-r:N//2+r, N//2-r:N//2+r] = 0.25 + 0.02 * rng.standard_normal((2*r, 2*r))

    for _ in range(n_steps):
        uvv  = u * v * v
        du   = Du * laplacian2d(u) - uvv + F * (1 - u)
        dv   = Dv * laplacian2d(v) + uvv - (F + k) * v
        u += dt * du
        v += dt * dv
    return v


# Three parameter sets give qualitatively different patterns
params = [
    dict(F=0.035, k=0.065, label='Spots'),
    dict(F=0.060, k=0.062, label='Stripes'),
    dict(F=0.025, k=0.060, label='Holes'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, p in zip(axes, params):
    print(f"Running Gray–Scott ({p['label']}) …")
    v_field = gray_scott(N=100, n_steps=5000, F=p['F'], k=p['k'])
    ax.imshow(v_field, cmap='inferno', interpolation='bilinear')
    ax.set_title(f"{p['label']}\nF={p['F']}, k={p['k']}", fontsize=11)
    ax.axis('off')

fig.suptitle('Gray–Scott Reaction–Diffusion (Turing Patterns)', fontsize=13)
plt.tight_layout()
plt.show()

## Forest Fire Model

A percolation-type CA that models wildfire spread and self-organised criticality:

| State | Meaning |
|-------|---------|
| 0 | Empty |
| 1 | Tree |
| 2 | Burning |

**Rules** (synchronous):
- Burning → Empty
- Tree with a burning neighbour → Burning
- Empty → Tree with probability $p$
- Tree → Burning spontaneously with probability $f \ll p$ (lightning)

In [ ]:
# ── Forest Fire CA ───────────────────────────────────────────────────────────

EMPTY, TREE, FIRE = 0, 1, 2

def forest_fire_step(grid: np.ndarray, p_grow: float = 0.04,
                     p_lightning: float = 0.0005,
                     rng: np.random.Generator = None) -> np.ndarray:
    """One step of the Drossel–Schwabl forest-fire model."""
    if rng is None:
        rng = np.random.default_rng()
    new = grid.copy()

    # Burning cells become empty
    new[grid == FIRE] = EMPTY

    # Trees with a burning neighbour catch fire
    burning_nbr = (
        np.roll(grid == FIRE, 1,  axis=0) |
        np.roll(grid == FIRE, -1, axis=0) |
        np.roll(grid == FIRE, 1,  axis=1) |
        np.roll(grid == FIRE, -1, axis=1)
    )
    new[(grid == TREE) & burning_nbr] = FIRE

    # Empty cells sprout trees
    new[(grid == EMPTY) & (rng.random(grid.shape) < p_grow)] = TREE

    # Lightning strikes trees
    new[(grid == TREE) & (rng.random(grid.shape) < p_lightning)] = FIRE

    return new


N = 100
rng_ff = np.random.default_rng(7)
grid_ff = rng_ff.integers(0, 2, size=(N, N)).astype(np.uint8)  # random trees

# Warm up
for _ in range(300):
    grid_ff = forest_fire_step(grid_ff, rng=rng_ff)

# Animate
cmap_ff = ListedColormap(['#3d2b1f', '#2d6a4f', '#e63946'])
fig_ff, ax_ff = plt.subplots(figsize=(6, 6))
im_ff = ax_ff.imshow(grid_ff, cmap=cmap_ff, vmin=0, vmax=2,
                     interpolation='nearest', animated=True)
ax_ff.axis('off')
ax_ff.set_title('Forest Fire CA')

def update_ff(frame):
    global grid_ff
    for _ in range(3):
        grid_ff = forest_fire_step(grid_ff, rng=rng_ff)
    im_ff.set_data(grid_ff)
    return [im_ff]

anim_ff = animation.FuncAnimation(fig_ff, update_ff,
                                   frames=80, interval=80, blit=True)
plt.close(fig_ff)
HTML(anim_ff.to_jshtml())

## Brian's Brain (Excitable Medium)

**Brian's Brain** is a 3-state CA modelling excitable media (neurons, cardiac tissue):

| State | Transition |
|-------|------------|
| **ON** (firing) | → Refractory (always) |
| **Refractory** | → OFF (always) |
| **OFF** | → ON if exactly 2 ON neighbours (else stays OFF) |

In [ ]:
# ── Brian's Brain ─────────────────────────────────────────────────────────────

OFF, ON, REFRACT = 0, 1, 2

def brians_brain_step(grid: np.ndarray) -> np.ndarray:
    new = grid.copy()
    # Count ON neighbours (Moore neighbourhood)
    on_nbr = sum(
        np.roll(np.roll(grid == ON, dr, axis=0), dc, axis=1)
        for dr in (-1, 0, 1)
        for dc in (-1, 0, 1)
        if (dr, dc) != (0, 0)
    )
    new[grid == ON]      = REFRACT
    new[grid == REFRACT] = OFF
    new[(grid == OFF) & (on_nbr == 2)] = ON
    return new


N = 100
brain = RNG.integers(0, 3, size=(N, N), dtype=np.uint8)
cmap_brain = ListedColormap(['#0f0f23', '#e9c46a', '#264653'])

fig_b, ax_b = plt.subplots(figsize=(6, 6))
im_b = ax_b.imshow(brain, cmap=cmap_brain, vmin=0, vmax=2,
                   interpolation='nearest', animated=True)
ax_b.axis('off')
ax_b.set_title("Brian's Brain (excitable medium)")

def update_brain(frame):
    global brain
    brain = brians_brain_step(brain)
    im_b.set_data(brain)
    return [im_b]

anim_brain = animation.FuncAnimation(fig_b, update_brain,
                                      frames=120, interval=50, blit=True)
plt.close(fig_b)
HTML(anim_brain.to_jshtml())

## Summary Table: CA in Physics

| CA Model | Physical System | Emergent PDE / Behaviour |
|----------|----------------|-------------------------|
| Random walk CA | Brownian motion | $\partial_t \rho = D \nabla^2 \rho$ |
| FHP / LBM | Viscous fluid | Navier–Stokes equations |
| Nagel–Schreckenberg | Road traffic | Burgers equation |
| Forest fire | Percolation / SOC | Power-law avalanche distribution |
| Gray–Scott | Turing morphogenesis | RD equations |
| Ising spin CA | Ferromagnetism | Phase transition, Glauber dynamics |
| Brian's Brain | Cardiac / neural excitation | FitzHugh–Nagumo PDE |